# 1. Extract Documents

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import re

In [2]:
# Set request headers
headers = {
    'User-Agent': 'Mozilla/5.0 (academic research)',
    'Accept-Language': 'en-US'
}

# Create a session and apply headers
session = requests.Session()
session.headers.update(headers)

# Create empty list to store document titles and links
document_links = []

# Base URL for FCC search results
base_url = 'https://www.fcc.gov/documents?field_edoc_doctype_target_id%5B0%5D=92&field_released_date_value=&exposed_from_date=&exposed_to_date=&page='

# Loop through search result pages
for page in range(0, 5):

    print('=' * 40)
    print(f'Search page {page}')

    # Build page URL
    page_url = base_url + str(page)

    # Request search result page
    response = session.get(page_url, timeout=30)
    print('Status code (Search page):', response.status_code)

    # Parse HTML
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')

    # Extract document titles and links
    for a in soup.select('div.headline-title a.title'):
        title = a.get_text(strip=True)
        link = urljoin('https://www.fcc.gov', a['href'])

        # Save title and link
        document_links.append({
            'title': title,
            'link': link
        })

        print(title, '>>>', link)

# Print total number of links
print('\n', 'Number of document links:', len(document_links))

Search page 0
Status code (Search page): 200
Trusty NAB Show Remarks >>> https://www.fcc.gov/document/trusty-nab-show-remarks
Trusty WTA Spring Educational Forum Remarks >>> https://www.fcc.gov/document/trusty-wta-spring-educational-forum-remarks
Commissioner Trusty Remarks at CLDP in Costa Rica >>> https://www.fcc.gov/document/commissioner-trusty-remarks-cldp-costa-rica
Commissioner Trusty Speaks at National Urban League >>> https://www.fcc.gov/document/commissioner-trusty-speaks-national-urban-league
Trusty Fiber Broadband Association Summit Remarks >>> https://www.fcc.gov/document/trusty-fiber-broadband-association-summit-remarks
Trusty State of the Net Keynote >>> https://www.fcc.gov/document/trusty-state-net-keynote
Gomez on Media Consolidation at SOTN >>> https://www.fcc.gov/document/gomez-media-consolidation-sotn
Trusty Incompas Policy Summit Keynote >>> https://www.fcc.gov/document/trusty-incompas-policy-summit-keynote
Trusty Hudson Institute Remarks >>> https://www.fcc.gov/doc

In [3]:
# Create output folder
output_folder = Path("fcc_documents")
output_folder.mkdir(exist_ok=True)

# Loop through collected document links
for i, doc in enumerate(document_links, start=1):
    
    print(f"Document_{i}: {doc['title']}")

    # Request document page
    doc_response = session.get(doc['link'], timeout=30)

    # Parse document page HTML
    doc_soup = BeautifulSoup(doc_response.text, 'html.parser')

    # Find text file link
    txt_tag = doc_soup.select_one('a.document-link.txt')   

    # Skip if no text file exists
    if txt_tag is None:
        print(">>> No text file found.")
        continue

    # Extract text file URL
    txt_url = txt_tag['href']

    # Download text file
    txt_response = session.get(txt_url, timeout=30)

    # Clean title for file name
    clean_title = re.sub(r'[\\/*?:"<>|]', "", doc['title'])
    clean_title = clean_title.replace(" ", "_")

    # Define output file path
    file_path = output_folder / f"{i}_{clean_title}.txt"

    # Save text file
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(txt_response.text)

Document_1: Trusty NAB Show Remarks
Document_2: Trusty WTA Spring Educational Forum Remarks
Document_3: Commissioner Trusty Remarks at CLDP in Costa Rica
Document_4: Commissioner Trusty Speaks at National Urban League
Document_5: Trusty Fiber Broadband Association Summit Remarks
Document_6: Trusty State of the Net Keynote
Document_7: Gomez on Media Consolidation at SOTN
Document_8: Trusty Incompas Policy Summit Keynote
Document_9: Trusty Hudson Institute Remarks
Document_10: Trusty International Institute of Communications Remarks
Document_11: Trusty 14th Americas Spectrum Management Keynote
Document_12: Trusty Media Institute Speech
Document_13: Trusty Remarks at Mobile World Congress
Document_14: Trusty Delivers Remarks at Infrastructure Vandalism Summit
Document_15: Build America: Unleashing America's Space Economy
Document_16: Commissioner Trusty Delivers Remarks at Free State Foundation
Document_17: Commissioner Trusty Delivers Remarks at CCA Annual Meeting
Document_18: Chairman C